# Steps 3 & 4: Train CodeT5-Small to detect vulnerable functions

Runs on **Google Colab with a free GPU**: *Runtime → Change runtime type → T4 GPU*.

We fine-tune **CodeT5-Small** (the paper's best small model) on our cleaned
DiverseVul data and test it on two splits:
- `random`: test functions come from the same projects as training (paper: ~45% F1)
- `unseen`: test functions come from 80 projects the model never saw (paper: ~9% F1)

**Step 4:** set `RENAME_IDS = True` to replace project-specific names with `VAR_1`, `FUNC_1`, ...
before training and testing. Does that help on unseen projects?

We need 4 runs in total (change the settings cell, then run cells 4–8):

| Run | SPLIT | RENAME_IDS |
|---|---|---|
| 1 | random | False |
| 2 | unseen | False |
| 3 | random | True |
| 4 | unseen | True |

## 1. Check the GPU

In [1]:
!nvidia-smi

Tue Oct 13 17:01:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code and install libraries
If the GitHub repo is **private**, Colab can't clone it. Easiest fix: make the repo public
(GitHub → Settings → Danger Zone → Change visibility).

**Start from a fresh runtime** (Runtime → Disconnect and delete runtime) so the
transformers version installed here is the one that gets loaded.

In [2]:
REPO_URL = "https://github.com/aashitagrawal/diversevul-vuln-detection.git"
!git clone -q {REPO_URL} project
%cd project
# transformers 5.x can't load the CodeT5 tokenizer, so we use the 4.x version
!pip install -q -r requirements.txt "transformers>=4.46,<5" datasets accelerate
import sys
sys.path.append("src")

/content/project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 476.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Download and clean the dataset (same scripts we ran on the laptop, ~5 min)

In [3]:
!python src/download_data.py
!python src/clean_dedup.py

Downloading...
From (original): https://drive.google.com/uc?id=12IWKhmLhq7qn5B_iXgn5YerOQtkH-6RG
From (redirected): https://drive.google.com/uc?id=12IWKhmLhq7qn5B_iXgn5YerOQtkH-6RG&confirm=t&uuid=daf252e1-83d4-47d7-a2de-9b64aeba3bdb
To: /content/project/data/diversevul_20230702.json
100% 737M/737M [00:17<00:00, 41.4MB/s]
Downloaded: data/diversevul_20230702.json
Files in data/: ['diversevul_20230702.json']
Loading data...
Normalizing functions...

=== Step 2: hidden duplicates (after removing whitespace + comments) ===
Functions before cleaning:            330492
Groups of duplicates found:           4397
Extra copies inside those groups:     4540
Groups with BOTH labels (conflicts):  900
  functions in conflicting groups:    1815
  of which labelled vulnerable:       901

Functions after cleaning:             325052
  vulnerable:                         18001
  safe:                               307051
Removed in total:                     5440

Example of a conflicting pair (same co

## 4. Settings

In [4]:
SPLIT = "random"      # "random" or "unseen"
RENAME_IDS = True    # Step 4: True = rename identifiers to VAR_1, FUNC_1, ...
MODEL_NAME = "Salesforce/codet5-small"
MAX_LEN = 256         # max tokens per function (longer functions are cut)
EPOCHS = 3            # paper used more; we're limited by free Colab GPU time
BATCH_SIZE = 16
LEARNING_RATE = 1e-4  # 5e-5 learned too slowly in 2 epochs (val F1 ~0.28)
FP16 = True           # faster on T4. If the loss shows "nan", set this to False

## 5. Load the data and split it

In [5]:
from load_data import load_diversevul
from splits import random_split, project_split, downsample_safe
from rename_ids import rename_identifiers

df = load_diversevul("data/diversevul_clean.json")
if RENAME_IDS:
    print("Renaming identifiers (~2 min)...")
    df["func"] = df["func"].apply(rename_identifiers)
if SPLIT == "random":
    train_df, val_df, test_df = random_split(df)
else:
    train_df, val_df, test_df = project_split(df)

# Keep all vulnerable functions + 3 safe ones per vulnerable one (faster training).
# Validation and test keep the real class balance.
train_df = downsample_safe(train_df, ratio=3)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(d):6d} functions, {d['target'].sum():5d} vulnerable")

Renaming identifiers (~2 min)...
train  57232 functions, 14308 vulnerable
val    32505 functions,  1836 vulnerable
test   32506 functions,  1857 vulnerable


## 6. Tokenize (turn code text into token ids)

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(d):
    # Some C code contains the text "</s>", which CodeT5 reads as its special
    # end-of-input token. Remove it so every function has exactly one.
    texts = [x.replace("</s>", "") for x in d["func"]]
    ds = Dataset.from_dict({"text": texts, "label": d["target"].tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/57232 [00:00<?, ? examples/s]

Map:   0%|          | 0/32505 [00:00<?, ? examples/s]

Map:   0%|          | 0/32506 [00:00<?, ? examples/s]

## 7. Train

In [7]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# The model also returns big internal tensors; keep only the prediction scores
# (otherwise evaluation runs out of memory)
def keep_logits(outputs, labels):
    return outputs[0] if isinstance(outputs, tuple) else outputs

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"f1": f1_score(labels, logits.argmax(-1))}

args = TrainingArguments(
    output_dir="models/codet5",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    fp16=FP16,
    eval_strategy="epoch",          # check validation F1 after every epoch
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,    # keep the epoch with the best validation F1
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics,
                  preprocess_logits_for_metrics=keep_logits)
trainer.train()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at Salesforce/codet5-small and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,F1
1,0.512800,0.251989,0.091719
2,0.502100,0.364151,0.229890
3,0.484800,0.323534,0.240854


There were missing keys in the checkpoint model loaded: ['transformer.encoder.embed_tokens.weight', 'transformer.decoder.embed_tokens.weight'].


TrainOutput(global_step=10731, training_loss=0.5028172842154975, metrics={'train_runtime': 2973.3824, 'train_samples_per_second': 57.744, 'train_steps_per_second': 3.609, 'total_flos': 1.1688362522984448e+16, 'train_loss': 0.5028172842154975, 'epoch': 3.0})

## 8. Pick the decision threshold, then test
We trained on 1 vulnerable : 3 safe, but real data is about 1 : 17. So "probability > 0.5"
is not the best cut-off. We pick the threshold that gives the best F1 on the
**validation** set, then use it on the **test** set (never tune on test).

In [8]:
import json
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

def vuln_probability(ds):
    logits = trainer.predict(ds).predictions
    return torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

val_prob = vuln_probability(val_ds)
thresholds = np.arange(0.05, 1.0, 0.05)
val_f1s = [f1_score(val_df["target"], val_prob > t) for t in thresholds]
best_t = float(thresholds[int(np.argmax(val_f1s))])
print(f"Best threshold on validation: {best_t:.2f} (val F1 = {max(val_f1s):.3f})")

test_prob = vuln_probability(test_ds)
y_true = test_df["target"].values
y_pred = (test_prob > best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

results = {
    "split": SPLIT,
    "rename_ids": RENAME_IDS,
    "threshold": best_t,
    "accuracy": accuracy_score(y_true, y_pred) * 100,
    "precision": precision_score(y_true, y_pred) * 100,
    "recall": recall_score(y_true, y_pred) * 100,
    "f1": f1_score(y_true, y_pred) * 100,
    "fpr": fp / (fp + tn) * 100,   # false positive rate: safe functions flagged as vulnerable
}
for k, v in results.items():
    print(f"{k:10s} {v:.2f}" if isinstance(v, float) else f"{k:10s} {v}")

RUN_NAME = f"{SPLIT}_{'renamed' if RENAME_IDS else 'baseline'}"
with open(f"results/{RUN_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)

Best threshold on validation: 0.50 (val F1 = 0.241)


split      random
rename_ids True
threshold  0.50
accuracy   89.59
precision  20.39
recall     28.33
f1         23.71
fpr        6.70


## 9. Done
Copy the printed numbers back to Claude (or download `results/<split>_<baseline|renamed>.json`
from the Files panel on the left). Then change the settings cell for the next run
(see the table at the top) and run cells 4–8 again. No need to re-download the data.